# Téléchargement des données d'entrée du papier (Tableau 1) — bassin de l'Oum Er-Rbia

Période : **1er janvier 2020 → 31 décembre 2025** (Introduction et Section 4 du papier). Exécuter d'abord la cellule *SETUP*, puis les cellules dans l'ordre ; chaque cellule peut être relancée, les fichiers déjà présents dans Drive sont ignorés.

| # | Variable du Tableau 1 | Asset GEE | Bande | Dossier Drive |
|---|---|---|---|---|
| 1 | Groundwater Storage (GWS) — cible | `NASA/GLDAS/V022/CLSM/G025/DA1D` | `GWS_tavg` | `PAPER_GWS_GLDAS` |
| 1 | Termes de l'Éq. (1) : TWS, RZSM, SWE, CI | idem | `TWS_tavg`, `SoilMoist_RZ_tavg`, `SWE_tavg`, `CanopInt_tavg` | `PAPER_TWS_GLDAS`, `PAPER_GLDAS_EQ1` |
| 2 | Précipitation (base de IDW et API) | `UCSB-CHG/CHIRPS/DAILY` | `precipitation` | `PAPER_PRECIP_CHIRPS` |
| 3 | Soil Moisture (SM) | `NASA/SMAP/SPL4SMGP/008` | `sm_surface` | `PAPER_SM_SMAP` |
| 3 | Root Zone Soil Moisture (RZSM) | `NASA/SMAP/SPL4SMGP/008` | `sm_rootzone` | `PAPER_RZSM_SMAP` |
| 4 | Évapotranspiration (ET) | `MODIS/061/MOD16A2GF` | `ET` | `PAPER_ET_MODIS` |
| 5 | NDVI | `MODIS/061/MOD13Q1` | `NDVI` | `PAPER_NDVI_MODIS` |
| 6 | Land Surface Temperature (LST) | `MODIS/061/MOD11A1` | `LST_Day_1km` | `PAPER_LST_MODIS` |
| 7 | Subsurface Runoff (Qsb) | `ECMWF/ERA5_LAND/DAILY_AGGR` | `sub_surface_runoff_sum` | `PAPER_QSB_ERA5LAND` |
| 8 | MNT (graphe D-8), Slope, TWI | `USGS/SRTMGL1_003` (+ `MERIT/Hydro/v1_0_1` pour l'aire drainée du TWI) | `elevation` | `PAPER_DEM_SRTM`, `PAPER_SLOPE_SRTM`, `PAPER_TWI_SRTM` |
| 9 | Distance to Rivers + réseau hydrographique | `WWF/HydroSHEDS/v1/FreeFlowingRivers` | — | `PAPER_DISTRIVERS_HYDROSHEDS` |

**Conventions de sortie** : un GeoTIFF par date, `float32`, EPSG:4326, valeurs brutes de l'asset (pas de facteur d'échelle appliqué), pixels sans donnée = `-9999`.

**Restent à calculer après téléchargement** (ce ne sont pas des téléchargements) : précipitation IDW (Éq. 3) et API (Éq. 4) à partir de CHIRPS, duplication des produits 8/16 jours sur les jours intermédiaires, reprojection bilinéaire sur la grille CHIRPS.

In [ ]:
# ============================================================
#  SETUP COMMUN — à exécuter UNE SEULE FOIS avant les cellules variables
# ============================================================
!pip install earthengine-api --quiet

import ee
from google.colab import drive
import glob
import math
import os
import random
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta, timezone

drive.mount('/content/drive')  # "already mounted" n'est pas une erreur

ee.Authenticate()
ee.Initialize(project='ee-anasseboutayeb1991')
print("✅ Earth Engine initialisé (projet ee-anasseboutayeb1991).")

# ---- Paramètres GLOBAUX partagés par toutes les variables ----
study_area = ee.FeatureCollection('users/anasseboutayeb1991/Regions')

# Période du papier (Introduction + Section 4) : 1er janvier 2020 → 31 décembre 2025
start_date = '2020-01-01'
end_date   = '2025-12-31'          # date de fin INCLUSE

# 'direct' : chaque image est téléchargée tout de suite dans le Drive monté (rapide, pas de file
#            d'attente GEE). Recommandé pour les séries temporelles.
# 'tasks'  : une tâche Export.image.toDrive par image, comme dans la version d'origine
#            (≈ 18 000 tâches sur 6 ans, soumises par lots de TASK_BATCH : beaucoup plus lent).
METHOD        = 'direct'
CLIP_TO_BASIN = True               # True : pixels hors bassin = NODATA. False : rectangle complet
                                   # (utile si les voisins hors bassin servent à l'interpolation).
NODATA        = -9999
N_WORKERS     = 8                  # téléchargements simultanés en mode 'direct'
TASK_BATCH    = 2000               # mode 'tasks' : GEE refuse plus de 3000 tâches en attente
DRIVE_ROOT    = '/content/drive/MyDrive'

start_py = datetime.strptime(start_date, '%Y-%m-%d')
end_py   = datetime.strptime(end_date,   '%Y-%m-%d')
# filterDate() EXCLUT sa borne de fin → on filtre jusqu'au lendemain de end_date.
end_excl = (end_py + timedelta(days=1)).strftime('%Y-%m-%d')

# Rectangle englobant du bassin (lon_min, lat_min, lon_max, lat_max)
_ring = study_area.geometry().bounds(1).getInfo()['coordinates'][0]
BBOX  = (min(p[0] for p in _ring), min(p[1] for p in _ring),
         max(p[0] for p in _ring), max(p[1] for p in _ring))
print(f"🗺️  Emprise du bassin : lon {BBOX[0]:.3f} → {BBOX[2]:.3f}, lat {BBOX[1]:.3f} → {BBOX[3]:.3f}")

M_PER_DEG = 111319.49079327357     # mètres par degré utilisés par GEE pour EPSG:4326


# ---------- Grille d'export ----------
def make_grid(collection, fallback_scale_m, pad=1):
    """
    Grille EPSG:4326 couvrant BBOX (+ `pad` pixels de marge), identique pour toutes les dates.
    - Produit déjà en lat/lon (GLDAS, CHIRPS, SMAP, ERA5-Land) : on reprend EXACTEMENT la grille
      native de l'asset (même pas, mêmes bords de pixels) → aucun rééchantillonnage.
    - Sinon (MODIS, sinusoïdal) : grille lat/lon au pas `fallback_scale_m` (plus proche voisin).
    """
    info  = ee.Image(collection.first()).getInfo()
    band0 = info['bands'][0]
    t     = band0.get('crs_transform')
    if band0.get('crs') == 'EPSG:4326' and t and t[1] == 0 and t[3] == 0:
        sx, x0, sy, y0, native = t[0], t[2], abs(t[4]), t[5], True
    else:
        sx = sy = fallback_scale_m / M_PER_DEG
        x0, y0, native = -180.0, 90.0, False

    eps = 1e-9
    xmin, ymin, xmax, ymax = BBOX
    k_left  = math.floor((xmin - x0) / sx + eps) - pad
    k_right = math.ceil((xmax - x0) / sx - eps) + pad
    k_bot   = math.floor((ymin - y0) / sy + eps) - pad
    k_top   = math.ceil((ymax - y0) / sy - eps) + pad
    return {
        'crs': 'EPSG:4326', 'sx': sx, 'sy': sy,
        'tx': x0 + k_left * sx,            # bord OUEST du premier pixel
        'ty': y0 + k_top * sy,             # bord NORD du premier pixel
        'width': k_right - k_left, 'height': k_top - k_bot,
        'native': native, 'nbands': len(info['bands']),
    }


def _crs_transform(grid):
    return [grid['sx'], 0, grid['tx'], 0, -grid['sy'], grid['ty']]


def _grid_region(grid):
    """Rectangle de la grille, rentré d'1/4 de pixel pour que l'export garde exactement width×height."""
    sx, sy = grid['sx'], grid['sy']
    return ee.Geometry.Rectangle(
        [grid['tx'] + sx / 4, grid['ty'] - grid['height'] * sy + sy / 4,
         grid['tx'] + grid['width'] * sx - sx / 4, grid['ty'] - sy / 4],
        proj='EPSG:4326', geodesic=False)


def _finalize(img):
    """float32, découpe éventuelle au bassin, pixels masqués → NODATA."""
    out = ee.Image(img).toFloat()
    if CLIP_TO_BASIN:
        out = out.clip(study_area)
    return out.unmask(NODATA, False)


# ---------- Dates disponibles ----------
def list_dates(collection):
    """Jours (UTC) pour lesquels la collection contient au moins une image → ['YYYY-MM-DD', ...]."""
    millis = collection.aggregate_array('system:time_start').getInfo()
    return sorted({datetime.fromtimestamp(m / 1000, tz=timezone.utc).strftime('%Y-%m-%d')
                   for m in millis})


def _report_coverage(dates, label):
    days  = [datetime.strptime(d, '%Y-%m-%d') for d in dates]
    steps = [(b - a).days for a, b in zip(days, days[1:])]
    step  = sorted(steps)[len(steps) // 2] if steps else 1
    print(f"📦 [{label}] {len(dates)} dates : {dates[0]} → {dates[-1]} (pas habituel : {step} j)")
    if (days[0] - start_py).days >= step:
        print(f"   ⚠️ La série commence le {dates[0]}, après le début demandé ({start_date}).")
    if (end_py - days[-1]).days >= step:
        print(f"   ⚠️ La série s'arrête le {dates[-1]}, avant la fin demandée ({end_date}).")
    holes = [(a, b) for a, b in zip(days, days[1:]) if (b - a).days > step]
    if holes:
        a, b = max(holes, key=lambda h: h[1] - h[0])
        print(f"   ⚠️ {len(holes)} trou(s) dans la série ; le plus long : "
              f"{a:%Y-%m-%d} → {b:%Y-%m-%d} ({(b - a).days} j).")


# ---------- Téléchargement direct d'une image ----------
def _fetch_tif(image, grid, path, retries=6):
    request = {
        'expression': image,
        'fileFormat': 'GEO_TIFF',
        'grid': {
            'dimensions': {'width': grid['width'], 'height': grid['height']},
            'affineTransform': {'scaleX': grid['sx'], 'shearX': 0, 'translateX': grid['tx'],
                                'shearY': 0, 'scaleY': -grid['sy'], 'translateY': grid['ty']},
            'crsCode': grid['crs'],
        },
    }
    for attempt in range(retries):
        try:
            data = ee.data.computePixels(request)
            if data[:4] not in (b'II*\x00', b'MM\x00*'):
                raise IOError("la réponse de GEE n'est pas un GeoTIFF")
            with open(path + '.part', 'wb') as f:
                f.write(data)
            os.replace(path + '.part', path)
            return
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(min(60, 2 ** attempt) + random.random())   # quota / réseau : on réessaie


# ---------- Export d'une série temporelle ----------
def export_timeseries(collection, band_or_image_fn, folder, file_tag,
                      scale, label, daily_mean=False):
    """
    Exporte une image par date disponible dans `collection` (déjà filtrée date + bande),
    sous le nom `<file_tag>_YYYYMMDD.tif` dans le dossier Drive `folder`.
    `band_or_image_fn(img)` renvoie l'image finale à exporter.
    `scale` (m) ne sert que si la grille native n'est pas en lat/lon (MODIS).
    `daily_mean=True` : moyenne des images du jour (produits infra-journaliers, ex. SMAP 3 h).
    Renvoie la liste des tâches GEE (vide en mode 'direct').
    """
    dates = list_dates(collection)
    if not dates:
        print(f"📦 [{label}] ⚠️ Aucune image sur la période/zone.")
        return []
    _report_coverage(dates, label)

    grid = make_grid(collection, scale)
    res  = (f"{grid['sx']:.6g}° (grille native)" if grid['native']
            else f"{grid['sx']:.6g}° (≈ {scale} m, rééchantillonné)")
    print(f"   Grille : {grid['width']} × {grid['height']} px, pas {res}")

    out_dir = os.path.join(DRIVE_ROOT, folder)
    os.makedirs(out_dir, exist_ok=True)

    def prefix(d):
        return f"{file_tag}_{d.replace('-', '')}"

    def path(d):
        return os.path.join(out_dir, prefix(d) + '.tif')

    def build(d):
        day = collection.filterDate(d, ee.Date(d).advance(1, 'day'))
        img = day.mean() if daily_mean else ee.Image(day.first())
        return _finalize(band_or_image_fn(img))

    todo = [d for d in dates if not (os.path.exists(path(d)) and os.path.getsize(path(d)) > 0)]
    print(f"   {len(dates) - len(todo)} déjà dans Drive, {len(todo)} à exporter → {out_dir}")
    if not todo:
        return []

    if METHOD == 'direct':
        size = grid['width'] * grid['height'] * grid['nbands'] * 4
        if size > 48e6 or max(grid['width'], grid['height']) > 32768:
            raise ValueError(f"[{label}] image trop grande pour le téléchargement direct "
                             f"({size / 1e6:.0f} Mo) : utiliser METHOD = 'tasks'.")
        # Première image seule et sans réessai : si la requête est invalide, on le voit tout de suite.
        _fetch_tif(build(todo[0]), grid, path(todo[0]), retries=1)
        failed = []
        with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
            futures = {pool.submit(_fetch_tif, build(d), grid, path(d)): d for d in todo[1:]}
            for i, fut in enumerate(as_completed(futures), start=2):
                try:
                    fut.result()
                except Exception as e:
                    failed.append((futures[fut], e))
                if i % 200 == 0 or i == len(todo):
                    print(f"   ➤ [{label}] {i}/{len(todo)}")
        if failed:
            failed.sort(key=lambda f: f[0])
            print(f"   ⚠️ [{label}] {len(failed)} échec(s) — relancer la cellule pour les reprendre. "
                  f"Premier : {failed[0][0]} → {failed[0][1]}")
        else:
            print(f"🎉 [{label}] Terminé.\n")
        return []

    tasks = []
    for i, d in enumerate(todo, start=1):
        task = ee.batch.Export.image.toDrive(
            image          = build(d),
            description    = prefix(d),
            folder         = folder,
            fileNamePrefix = prefix(d),
            region         = _grid_region(grid),
            crs            = grid['crs'],
            crsTransform   = _crs_transform(grid),
            maxPixels      = 1e13
        )
        task.start()
        tasks.append(task)
        if i % 100 == 0 or i == len(todo):
            print(f"   ➤ [{i}/{len(todo)}] Soumis : {prefix(d)}")
        if i % TASK_BATCH == 0 and i < len(todo):
            monitor(tasks[-TASK_BATCH:], label)     # on vide la file avant de soumettre la suite
    return tasks


# ---------- Export d'une couche statique (toujours par tâche GEE : rasters 30 m volumineux) ----------
def export_static(image, folder, name, scale=30):
    out_dir = os.path.join(DRIVE_ROOT, folder)
    if glob.glob(os.path.join(out_dir, name + '*.tif')):
        print(f"📦 [{name}] déjà dans Drive ({out_dir}) — ignoré.")
        return None
    task = ee.batch.Export.image.toDrive(
        image          = _finalize(image),
        description    = name,
        folder         = folder,
        fileNamePrefix = name,
        region         = study_area.geometry(),
        scale          = scale,
        crs            = 'EPSG:4326',
        maxPixels      = 1e13
    )
    task.start()
    print(f"📦 [{name}] Tâche soumise (dossier {folder})")
    return task


# ---------- Suivi des tâches ----------
def monitor(tasks, label):
    tasks = [t for t in tasks if t is not None]
    if not tasks:
        print(f"🏁 [{label}] Aucune tâche GEE à suivre.\n")
        return
    print(f"\n⏳ [{label}] Vérification toutes les 30 s…\n")
    while True:
        states    = [t.status().get('state') for t in tasks]
        completed = states.count('COMPLETED')
        failed    = states.count('FAILED') + states.count('CANCELLED')
        running   = states.count('RUNNING')
        ready     = states.count('READY')
        print(f"🔍 [{label}] COMPLETED={completed}, RUNNING={running}, READY={ready}, FAILED={failed}")
        if failed > 0:
            print("   ⚠️ Tâche(s) en échec — vérifie l'onglet 'Tasks' de GEE.")
        if completed + failed == len(tasks):
            print(f"🎉 [{label}] Terminé.\n")
            break
        time.sleep(30)

In [ ]:
# ============================================================
#  VARIABLE 1 : GLDAS-2.2 (assimilation GRACE) — GWS cible + termes de l'Éq.(1)
#  Asset : NASA/GLDAS/V022/CLSM/G025/DA1D  |  0.25°  |  Quotidien
#  Éq.(1) : GWS = TWS − RZSM − SWE − CI
#  Dossiers Drive : PAPER_GWS_GLDAS, PAPER_TWS_GLDAS, PAPER_GLDAS_EQ1
#  🔑 Export sur la grille NATIVE 0.25° (l'ancien scale=25000 rééchantillonnait à ≈ 0.2246°).
#  ℹ️ Le papier n'utilise GLDAS que jusqu'au 2025-06-30 (juil.–déc. 2025 = période prédite).
#     Le catalogue va aujourd'hui plus loin : ces mois sont téléchargés aussi et peuvent servir
#     à valider la prédiction. Pour reproduire strictement le papier : GLDAS_END = '2025-07-01'.
# ============================================================
GLDAS_END  = end_excl        # borne de fin EXCLUE

FOLDER_GWS = 'PAPER_GWS_GLDAS'
FOLDER_TWS = 'PAPER_TWS_GLDAS'
FOLDER_EQ1 = 'PAPER_GLDAS_EQ1'

gldas = (ee.ImageCollection("NASA/GLDAS/V022/CLSM/G025/DA1D")
         .filterBounds(study_area)
         .filterDate(start_date, GLDAS_END))

def build_gldas(img):
    # Valeurs BRUTES (mm ; kg/m² ≡ mm d'eau), aucune conversion.
    return img.copyProperties(img, ['system:time_start'])

# 1a — GWS_tavg : variable cible du Tableau 1
tasks_gws = export_timeseries(gldas.select('GWS_tavg'), build_gldas, FOLDER_GWS,
                              'PAPER_GWS_daily', scale=27830, label='GWS')
monitor(tasks_gws, 'GWS')

# 1b — TWS_tavg (comme dans la version d'origine du notebook)
tasks_tws = export_timeseries(gldas.select('TWS_tavg'), build_gldas, FOLDER_TWS,
                              'PAPER_TWS_daily', scale=27830, label='TWS')
monitor(tasks_tws, 'TWS')

# 1c — Autres termes de l'Éq.(1) : un GeoTIFF à 3 bandes par jour
#      bande 1 = SoilMoist_RZ_tavg (RZSM), bande 2 = SWE_tavg (SWE), bande 3 = CanopInt_tavg (CI)
EQ1_BANDS = ['SoilMoist_RZ_tavg', 'SWE_tavg', 'CanopInt_tavg']
tasks_eq1 = export_timeseries(gldas.select(EQ1_BANDS), build_gldas, FOLDER_EQ1,
                              'PAPER_GLDAS_EQ1_daily', scale=27830, label='GLDAS-EQ1')
monitor(tasks_eq1, 'GLDAS-EQ1')

In [ ]:
# ============================================================
#  VARIABLE 2 : Précipitation CHIRPS
#  Asset : UCSB-CHG/CHIRPS/DAILY  |  0.05°  |  Quotidien
#  Dossier Drive : PAPER_PRECIP_CHIRPS
#  (NB : la précipitation IDW de l'Éq.(3) et l'API de l'Éq.(4) se calculent ENSUITE ;
#        ici on exporte la précip brute, sur la grille CHIRPS native = grille cible 0.05°)
# ============================================================
FOLDER_PR = 'PAPER_PRECIP_CHIRPS'

chirps = (ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
          .filterBounds(study_area)
          .filterDate(start_date, end_excl)
          .select('precipitation'))

def build_pr(img):
    return img  # valeur brute en mm/jour

tasks_pr = export_timeseries(chirps, build_pr, FOLDER_PR,
                             'PAPER_PRECIP_daily', scale=5566, label='CHIRPS')
monitor(tasks_pr, 'CHIRPS')

In [ ]:
# ============================================================
#  VARIABLES 3 : Soil Moisture (SM) et Root Zone Soil Moisture (RZSM)
#  Asset : NASA/SMAP/SPL4SMGP/008  |  ≈ 11 km  |  3-horaire (8 images par jour)
#  Dossiers Drive : PAPER_SM_SMAP, PAPER_RZSM_SMAP
#  🔑 Le jeu de données du papier est quotidien : on exporte la MOYENNE des images
#     3-horaires de chaque jour (m³/m³). Le papier ne précise pas l'agrégation utilisée.
# ============================================================
FOLDER_SM   = 'PAPER_SM_SMAP'
FOLDER_RZSM = 'PAPER_RZSM_SMAP'

smap = (ee.ImageCollection("NASA/SMAP/SPL4SMGP/008")
        .filterBounds(study_area)
        .filterDate(start_date, end_excl))

def build_smap(img):
    return img  # fraction volumique m³/m³

# 3a — SM : humidité de surface (0–5 cm)
tasks_sm = export_timeseries(smap.select('sm_surface'), build_smap, FOLDER_SM,
                             'PAPER_SM_daily', scale=11000, label='SM', daily_mean=True)
monitor(tasks_sm, 'SM')

# 3b — RZSM : humidité de la zone racinaire (0–100 cm)
tasks_rzsm = export_timeseries(smap.select('sm_rootzone'), build_smap, FOLDER_RZSM,
                               'PAPER_RZSM_daily', scale=11000, label='RZSM', daily_mean=True)
monitor(tasks_rzsm, 'RZSM')

In [ ]:
# ============================================================
#  VARIABLE 4 : Évapotranspiration (ET)
#  Asset : MODIS/061/MOD16A2GF  |  500 m  |  8 jours (fréquence native)
#  Dossier Drive : PAPER_ET_MODIS
#  🔑 Échelle : ET est en (kg/m²/8jours) * 0.1
#  ⚠️ Plage valide : −32767 … 32700. Au-delà (32761–32767) ce sont des codes de remplissage
#     du produit (sol nu, urbain, eau, neige…) : à écarter avant tout calcul.
# ============================================================
FOLDER_ET = 'PAPER_ET_MODIS'

modis_et = (ee.ImageCollection("MODIS/061/MOD16A2GF")
            .filterBounds(study_area)
            .filterDate(start_date, end_excl)
            .select('ET'))

def build_et(img):
    # Valeur BRUTE conservée
    return img.copyProperties(img, ['system:time_start'])

    # 🔁 OPTION conversion (kg/m²/8j réels, codes de remplissage masqués) — décommenter si besoin :
    # return (img.updateMask(img.lte(32700))
    #            .multiply(0.1)
    #            .rename('ET_real')
    #            .copyProperties(img, ['system:time_start']))

tasks_et = export_timeseries(modis_et, build_et, FOLDER_ET,
                             'PAPER_ET_8day', scale=500, label='ET')
monitor(tasks_et, 'ET')

In [ ]:
# ============================================================
#  VARIABLE 5 : NDVI
#  Asset : MODIS/061/MOD13Q1  |  250 m  |  16 jours (fréquence native)
#  Dossier Drive : PAPER_NDVI_MODIS
#  🔑 Échelle : NDVI est stocké * 0.0001
# ============================================================
FOLDER_NDVI = 'PAPER_NDVI_MODIS'

modis_ndvi = (ee.ImageCollection("MODIS/061/MOD13Q1")
              .filterBounds(study_area)
              .filterDate(start_date, end_excl)
              .select('NDVI'))

def build_ndvi(img):
    # Valeur BRUTE conservée
    return img.copyProperties(img, ['system:time_start'])

    # 🔁 OPTION conversion (NDVI réel entre -1 et 1) — décommenter si besoin :
    # return (img.multiply(0.0001)
    #            .rename('NDVI_real')
    #            .copyProperties(img, ['system:time_start']))

tasks_ndvi = export_timeseries(modis_ndvi, build_ndvi, FOLDER_NDVI,
                               'PAPER_NDVI_16day', scale=250, label='NDVI')
monitor(tasks_ndvi, 'NDVI')

In [ ]:
# ============================================================
#  VARIABLE 6 : Land Surface Temperature (LST)
#  Asset : MODIS/061/MOD11A1  |  1 km  |  Quotidien
#  Dossier Drive : PAPER_LST_MODIS
#  🔑 Échelle : LST_Day_1km est stocké en Kelvin * 50 (facteur 0.02)
#  ⚠️ Les pixels nuageux sont absents du produit → NODATA dans les fichiers (trous fréquents).
# ============================================================
FOLDER_LST = 'PAPER_LST_MODIS'

modis_lst = (ee.ImageCollection("MODIS/061/MOD11A1")
             .filterBounds(study_area)
             .filterDate(start_date, end_excl)
             .select('LST_Day_1km'))

def build_lst(img):
    # Valeur BRUTE conservée
    return img.copyProperties(img, ['system:time_start'])

    # 🔁 OPTION conversion (Kelvin) — décommenter si besoin :
    # return (img.multiply(0.02)
    #            .rename('LST_K')
    #            .copyProperties(img, ['system:time_start']))

tasks_lst = export_timeseries(modis_lst, build_lst, FOLDER_LST,
                              'PAPER_LST_daily', scale=1000, label='LST')
monitor(tasks_lst, 'LST')

In [ ]:
# ============================================================
#  VARIABLE 7 : Subsurface Runoff (Qsb)
#  Asset : ECMWF/ERA5_LAND/DAILY_AGGR  |  0.1°  |  Quotidien
#  Dossier Drive : PAPER_QSB_ERA5LAND
#  🔑 sub_surface_runoff_sum est un cumul journalier en MÈTRES d'eau.
# ============================================================
FOLDER_QSB = 'PAPER_QSB_ERA5LAND'

era5_qsb = (ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
            .filterBounds(study_area)
            .filterDate(start_date, end_excl)
            .select('sub_surface_runoff_sum'))

def build_qsb(img):
    # Valeur BRUTE conservée (m/jour)
    return img.copyProperties(img, ['system:time_start'])

    # 🔁 OPTION conversion (mm/jour) — décommenter si besoin :
    # return (img.multiply(1000)
    #            .rename('Qsb_mm')
    #            .copyProperties(img, ['system:time_start']))

tasks_qsb = export_timeseries(era5_qsb, build_qsb, FOLDER_QSB,
                              'PAPER_QSB_daily', scale=11132, label='QSB')
monitor(tasks_qsb, 'QSB')

In [ ]:
# ============================================================
#  VARIABLES 8 : couches STATIQUES dérivées du SRTM — MNT, Slope, TWI
#  Asset : USGS/SRTMGL1_003  |  30 m
#  Dossiers Drive : PAPER_DEM_SRTM, PAPER_SLOPE_SRTM, PAPER_TWI_SRTM
#  Pas de boucle temporelle : une image par couche (tâches Export.toDrive, fichiers volumineux ;
#  GEE peut découper chaque couche en plusieurs tuiles « -0000000000-0000000000.tif »).
#
#  TWI = ln(α / tan β)  — Éq.(5)
#   • β : pente SRTM 30 m.
#   • α : aire drainée spécifique. GEE ne sait pas calculer une accumulation de flux ; on prend
#         l'aire drainée amont `upa` de MERIT Hydro (≈ 90 m) divisée par la largeur de contour.
#     ⚠️ Le papier ne décrit pas son calcul de α : c'est une approximation. Pour un TWI
#        100 % SRTM à 30 m, calculer l'accumulation D8 en local à partir de PAPER_DEM_static
#        (WhiteboxTools, pysheds…).
# ============================================================
FOLDER_DEM   = 'PAPER_DEM_SRTM'
FOLDER_SLOPE = 'PAPER_SLOPE_SRTM'
FOLDER_TWI   = 'PAPER_TWI_SRTM'

srtm  = ee.Image("USGS/SRTMGL1_003")
dem   = srtm.select('elevation')                 # altitude en m (sert aussi au graphe D-8)
slope = ee.Terrain.slope(srtm).rename('slope')   # pente en degrés

CONTOUR_W     = 90.0    # m — largeur de contour ≈ taille d'un pixel MERIT Hydro
MIN_SLOPE_DEG = 0.1     # évite tan(0) = 0 sur les zones plates
upa_m2 = ee.Image("MERIT/Hydro/v1_0_1").select('upa').multiply(1e6)      # km² → m²
tan_b  = slope.max(ee.Image.constant(MIN_SLOPE_DEG)).multiply(math.pi / 180).tan()
twi    = upa_m2.divide(CONTOUR_W).divide(tan_b).log().rename('TWI')

tasks_static = [
    export_static(dem,   FOLDER_DEM,   'PAPER_DEM_static'),
    export_static(slope, FOLDER_SLOPE, 'PAPER_SLOPE_static'),
    export_static(twi,   FOLDER_TWI,   'PAPER_TWI_static'),
]
monitor(tasks_static, 'SRTM')

In [ ]:
# ============================================================
#  VARIABLE 9 : Distance to Rivers — couche STATIQUE
#  Asset : WWF/HydroSHEDS/v1/FreeFlowingRivers (réseau hydrographique HydroSHEDS, vecteur)
#  Dossier Drive : PAPER_DISTRIVERS_HYDROSHEDS
#  Sorties : PAPER_DISTRIVERS_static (distance euclidienne en m, 30 m)
#            PAPER_RIVERS_hydrosheds.geojson (tronçons du bassin, pour contrôle / autre usage)
# ============================================================
FOLDER_RIV = 'PAPER_DISTRIVERS_HYDROSHEDS'

rivers = (ee.FeatureCollection("WWF/HydroSHEDS/v1/FreeFlowingRivers")
          .filterBounds(study_area))
# 🔁 OPTION : ne garder que les cours d'eau principaux (RIV_ORD petit = gros débit) :
# rivers = rivers.filter(ee.Filter.lte('RIV_ORD', 6))
print(f"🌊 Tronçons HydroSHEDS qui intersectent le bassin : {rivers.size().getInfo()}")

# Distance (m) de chaque pixel au tronçon le plus proche ; masqué au-delà de 100 km.
dist_rivers = rivers.distance(searchRadius=100000, maxError=10).rename('dist_river_m')

task_dist = export_static(dist_rivers, FOLDER_RIV, 'PAPER_DISTRIVERS_static')

task_riv = None
if not os.path.exists(os.path.join(DRIVE_ROOT, FOLDER_RIV, 'PAPER_RIVERS_hydrosheds.geojson')):
    task_riv = ee.batch.Export.table.toDrive(
        collection     = rivers,
        description    = 'PAPER_RIVERS_hydrosheds',
        folder         = FOLDER_RIV,
        fileNamePrefix = 'PAPER_RIVERS_hydrosheds',
        fileFormat     = 'GeoJSON'
    )
    task_riv.start()
    print(f"📦 [PAPER_RIVERS_hydrosheds] Tâche soumise (dossier {FOLDER_RIV})")

monitor([task_dist, task_riv], 'RIVERS')

In [ ]:
# ============================================================
#  CONTRÔLE : inventaire des fichiers présents dans Drive
#  (les exports par tâche GEE peuvent mettre une ou deux minutes à apparaître dans /content/drive)
# ============================================================
n_days = (end_py - start_py).days + 1
INVENTORY = [
    # (dossier, motif, nombre attendu ou None)
    ('PAPER_GWS_GLDAS',             'PAPER_GWS_daily_*.tif',          n_days),
    ('PAPER_TWS_GLDAS',             'PAPER_TWS_daily_*.tif',          n_days),
    ('PAPER_GLDAS_EQ1',             'PAPER_GLDAS_EQ1_daily_*.tif',    n_days),
    ('PAPER_PRECIP_CHIRPS',         'PAPER_PRECIP_daily_*.tif',       n_days),
    ('PAPER_SM_SMAP',               'PAPER_SM_daily_*.tif',           n_days),
    ('PAPER_RZSM_SMAP',             'PAPER_RZSM_daily_*.tif',         n_days),
    ('PAPER_LST_MODIS',             'PAPER_LST_daily_*.tif',          n_days),
    ('PAPER_QSB_ERA5LAND',          'PAPER_QSB_daily_*.tif',          n_days),
    ('PAPER_ET_MODIS',              'PAPER_ET_8day_*.tif',            None),
    ('PAPER_NDVI_MODIS',            'PAPER_NDVI_16day_*.tif',         None),
    ('PAPER_DEM_SRTM',              'PAPER_DEM_static*.tif',          None),
    ('PAPER_SLOPE_SRTM',            'PAPER_SLOPE_static*.tif',        None),
    ('PAPER_TWI_SRTM',              'PAPER_TWI_static*.tif',          None),
    ('PAPER_DISTRIVERS_HYDROSHEDS', 'PAPER_DISTRIVERS_static*.tif',   None),
    ('PAPER_DISTRIVERS_HYDROSHEDS', 'PAPER_RIVERS_hydrosheds*',       None),
]

print(f"Période {start_date} → {end_date} : {n_days} jours\n")
for folder, pattern, expected in INVENTORY:
    files = sorted(glob.glob(os.path.join(DRIVE_ROOT, folder, pattern)))
    size  = sum(os.path.getsize(f) for f in files) / 1e6
    if not files:
        status = '❌ aucun fichier'
    elif expected is not None and len(files) != expected:
        status = f'⚠️ {len(files)} / {expected} attendus'
    else:
        status = f'✅ {len(files)}'
    print(f"{status:<28} {size:9.1f} Mo   {folder}/{pattern}")